# Data Analyst Assessment — Olist Dataset Processing
**Candidate:** Anshu Kumari | Meerut Institute of Technology

This notebook does the actual work behind **Question 3 (Data Processing)** of the assessment:
loads the raw Olist CSVs, cleans/joins them, and prints evidence you can copy straight into
the **Q3** worksheet and the **Processed Data** worksheet of your Google Sheet.

**How to use this in Google Colab:**
1. Open [colab.research.google.com](https://colab.research.google.com) → File → Upload notebook → upload this `.ipynb`.
2. Run the cells top to bottom (Runtime → Run all).
3. Cell 2 downloads the dataset straight from Kaggle using `kagglehub` (no manual download needed,
   but you do need a free Kaggle account the first time it asks you to sign in).
4. At the end, a file called `processed_data.csv` is created — download it (left sidebar → Files →
   right-click → Download) and paste/import it into the **Processed Data** tab of your Google Sheet.
5. The printed "Q3 EVIDENCE" blocks near the end give you the real before/after numbers to fill into
   your Q3 explanation table.


## 1. Setup & Download the Dataset

In [ ]:
# Run once per Colab session
!pip install kagglehub pandas numpy -q

import kagglehub
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Downloads the Olist dataset from Kaggle (asks you to authenticate the first time)
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print("Dataset downloaded to:", path)

import os
print(os.listdir(path))


## 2. Load the Raw Files

The dataset has 9 related CSVs. We load each one separately first, so we can inspect
and clean them before joining.

In [ ]:
orders      = pd.read_csv(f"{path}/olist_orders_dataset.csv")
items       = pd.read_csv(f"{path}/olist_order_items_dataset.csv")
payments    = pd.read_csv(f"{path}/olist_order_payments_dataset.csv")
reviews     = pd.read_csv(f"{path}/olist_order_reviews_dataset.csv")
products    = pd.read_csv(f"{path}/olist_products_dataset.csv")
customers   = pd.read_csv(f"{path}/olist_customers_dataset.csv")
sellers     = pd.read_csv(f"{path}/olist_sellers_dataset.csv")
geolocation = pd.read_csv(f"{path}/olist_geolocation_dataset.csv")
cat_translation = pd.read_csv(f"{path}/product_category_name_translation.csv")

raw_tables = {
    "orders": orders, "items": items, "payments": payments, "reviews": reviews,
    "products": products, "customers": customers, "sellers": sellers,
    "geolocation": geolocation, "cat_translation": cat_translation,
}

print("RAW ROW COUNTS")
for name, df in raw_tables.items():
    print(f"  {name:<15} rows={len(df):>7}   cols={df.shape[1]}")


## 3. Initial Data-Quality Inspection (before cleaning)

In [ ]:
print("--- Missing values per table (top offenders) ---")
for name, df in raw_tables.items():
    miss = df.isna().sum()
    miss = miss[miss > 0]
    if len(miss):
        print(f"\n{name}:")
        print(miss)

print("\n--- Duplicate rows per table ---")
for name, df in raw_tables.items():
    print(f"  {name:<15} duplicate rows: {df.duplicated().sum()}")

print("\n--- Data types (orders table) ---")
print(orders.dtypes)


## 4. Cleaning Decision 1 — Flag (not delete) undelivered/cancelled orders

`order_delivered_customer_date` is missing whenever an order was cancelled or is still
in transit. Dropping these rows would hide the cancellation rate. We keep them and add
an `is_delivered` flag instead.

In [ ]:
before_rows = len(orders)
missing_delivery_date = orders["order_delivered_customer_date"].isna().sum()

orders["is_delivered"] = orders["order_delivered_customer_date"].notna()

print("Q3 EVIDENCE — Cleaning Decision 1: Missing delivery dates")
print(f"  Total orders                         : {before_rows}")
print(f"  Orders missing delivered_customer_date: {missing_delivery_date} "
      f"({missing_delivery_date/before_rows:.1%})")
print(f"  Orders kept (is_delivered flag added) : {before_rows}  <-- no rows dropped")
print(orders["order_status"].value_counts())


## 5. Cleaning Decision 2 — Standardize product categories (Portuguese → English)

Raw category names are in Portuguese with some inconsistent casing. We translate them
using the official mapping file and clean up text formatting.

In [ ]:
raw_category_count = products["product_category_name"].nunique(dropna=True)

products["product_category_name"] = (
    products["product_category_name"].astype(str).str.strip().str.lower()
)
cat_translation["product_category_name"] = (
    cat_translation["product_category_name"].astype(str).str.strip().str.lower()
)

products = products.merge(cat_translation, on="product_category_name", how="left")
products["category_english"] = (
    products["product_category_name_english"].fillna(products["product_category_name"])
)

clean_category_count = products["category_english"].nunique(dropna=True)
untranslated = products["product_category_name_english"].isna().sum()

print("Q3 EVIDENCE — Cleaning Decision 2: Category standardization")
print(f"  Distinct raw (Portuguese) categories : {raw_category_count}")
print(f"  Distinct clean (English) categories  : {clean_category_count}")
print(f"  Products with no English translation found (kept as-is): {untranslated}")


## 6. Cleaning Decision 3 — Correct data types & create calculated fields

Convert all date columns to real datetimes, then derive the fields the whole analysis
depends on: `delivery_days`, `delivery_delay_days`, `order_total_value`, `is_late`.

In [ ]:
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
before_dtype = str(orders["order_purchase_timestamp"].dtype)
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")
after_dtype = str(orders["order_purchase_timestamp"].dtype)

orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

orders["is_late"] = orders["delivery_delay_days"] > 0

print("Q3 EVIDENCE — Cleaning Decision 3: Date types & calculated fields")
print(f"  order_purchase_timestamp dtype before: {before_dtype}")
print(f"  order_purchase_timestamp dtype after : {after_dtype}")
print(f"  delivery_days      -> min={orders['delivery_days'].min()}, "
      f"max={orders['delivery_days'].max()}, mean={orders['delivery_days'].mean():.1f}")
print(f"  Orders flagged late (is_late=True)    : {orders['is_late'].sum()} "
      f"({orders['is_late'].mean():.1%} of delivered orders)")


## 7. Remove Duplicates

In [ ]:
dupe_counts = {}
for name, df in [("items", items), ("payments", payments), ("reviews", reviews)]:
    before = len(df)
    df.drop_duplicates(inplace=True)
    after = len(df)
    dupe_counts[name] = before - after

print("Duplicate rows removed:")
for name, n in dupe_counts.items():
    print(f"  {name:<10}: {n} duplicate rows removed")


## 8. Identify Outliers / Unusual Records

In [ ]:
order_value = items.groupby("order_id").agg(
    item_price=("price", "sum"), freight_value=("freight_value", "sum")
).reset_index()
order_value["order_total_value"] = order_value["item_price"] + order_value["freight_value"]
order_value["freight_pct_of_price"] = order_value["freight_value"] / order_value["item_price"].replace(0, np.nan)

freight_over_price = (order_value["freight_value"] > order_value["item_price"]).sum()
bad_delivery_days = (orders["delivery_days"] < 0).sum()

print("Q3 EVIDENCE — Outliers / unusual records identified")
print(f"  Orders where freight_value > item price       : {freight_over_price}")
print(f"  Orders with negative delivery_days (timestamp errors): {bad_delivery_days}")
print("  These were flagged (kept, not deleted) via 'freight_pct_of_price' and 'is_delivered' flags")


## 9. Join Everything into One Analysis-Ready Fact Table

Joins: orders → items → products → sellers → customers → payments (agg) → reviews (agg).
Grain: one row per order item.

In [ ]:
payments_agg = payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    payment_installments_max=("payment_installments", "max"),
    payment_type_main=("payment_type", lambda x: x.mode().iat[0] if not x.mode().empty else np.nan),
).reset_index()

reviews_agg = reviews.groupby("order_id").agg(
    review_score=("review_score", "mean"),
).reset_index()
reviews_agg["review_bucket"] = pd.cut(
    reviews_agg["review_score"], bins=[0, 2, 3, 5],
    labels=["Negative (1-2)", "Neutral (3)", "Positive (4-5)"]
)

region_map = {
    "AC":"North","AP":"North","AM":"North","PA":"North","RO":"North","RR":"North","TO":"North",
    "AL":"Northeast","BA":"Northeast","CE":"Northeast","MA":"Northeast","PB":"Northeast",
    "PE":"Northeast","PI":"Northeast","RN":"Northeast","SE":"Northeast",
    "DF":"Central-West","GO":"Central-West","MT":"Central-West","MS":"Central-West",
    "ES":"Southeast","MG":"Southeast","RJ":"Southeast","SP":"Southeast",
    "PR":"South","RS":"South","SC":"South",
}
customers["customer_region"] = customers["customer_state"].map(region_map)

fact = (
    orders
    .merge(items, on="order_id", how="left")
    .merge(products[["product_id", "category_english"]], on="product_id", how="left")
    .merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left")
    .merge(customers[["customer_id", "customer_state", "customer_region"]], on="customer_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_agg, on="order_id", how="left")
)

fact["order_item_value"] = fact["price"] + fact["freight_value"]

print("Q3 EVIDENCE — Final joined table")
print(f"  Rows in final fact table   : {len(fact)}")
print(f"  Columns in final fact table: {fact.shape[1]}")
print(f"  Unique orders covered      : {fact['order_id'].nunique()}")
fact.head(3)


## 10. Category & Aggregation Tables (useful for Q4 insights + dashboard)

In [ ]:
category_summary = (
    fact.groupby("category_english")
    .agg(orders=("order_id", "nunique"), revenue=("order_item_value", "sum"))
    .sort_values("revenue", ascending=False)
    .reset_index()
)
print("Top 10 categories by revenue:")
display(category_summary.head(10))

state_summary = (
    fact.groupby("customer_state")
    .agg(orders=("order_id", "nunique"), revenue=("order_item_value", "sum"))
    .sort_values("revenue", ascending=False)
    .reset_index()
)
print("\nTop 10 states by revenue:")
display(state_summary.head(10))

delay_bucket = pd.cut(
    orders["delivery_delay_days"], bins=[-9999, 0, 3, 7, 9999],
    labels=["On-Time", "1-3 Days Late", "4-7 Days Late", "8+ Days Late"]
)
delay_vs_review = (
    orders.assign(delay_bucket=delay_bucket)
    .merge(reviews_agg[["order_id", "review_score"]], on="order_id", how="left")
    .groupby("delay_bucket")["review_score"].mean()
)
print("\nAverage review score by delivery-delay bucket:")
print(delay_vs_review)


## 11. Export the Processed, Analysis-Ready Dataset

In [ ]:
fact.to_csv("processed_data.csv", index=False)
category_summary.to_csv("category_summary.csv", index=False)
state_summary.to_csv("state_summary.csv", index=False)

print("Saved: processed_data.csv, category_summary.csv, state_summary.csv")
print("\nDownload these from the Colab file browser (left sidebar) and:")
print(" 1. Import processed_data.csv into the 'Processed Data' worksheet of your Google Sheet")
print(" 2. Use category_summary.csv / state_summary.csv as source data for your Looker Studio dashboard")


## 12. Summary Table for Your Q3 Worksheet

Copy the printed numbers above into this table format in your **Q3** worksheet:

| Change | Why was it necessary? | What would happen if you didn't do it? |
|---|---|---|
| Flagged (not deleted) orders with missing `delivered_customer_date` | These are cancelled/in-transit orders — real business signal, not an error | Dropping rows would hide the true cancellation rate and inflate on-time delivery % |
| Translated & standardized product categories (PT → EN) | Raw categories had 70+ inconsistent Portuguese labels | Category-level revenue would fragment into near-duplicate categories |
| Converted date columns to datetime & derived `delivery_days` / `delivery_delay_days` | Needed to test the late-delivery vs. review-score hypothesis | Delivery-performance analysis would be impossible on raw text dates |

Fill in the *italic* numbers above with your own printed output — they'll differ very slightly
depending on the exact Kaggle dataset version you download.
